In [1]:
import cv2 as cv
import caer


import tensorflow as tf
from keras import layers
from keras import models
from keras import optimizers
from keras import saving
from keras.utils import image_dataset_from_directory #это генератор исходных данных, обязательно ознакомиться с документацией!
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
#from tensorflow import data as tf_data
from keras.utils import plot_model
import keras
from IPython.display import Image, display
from tensorflow.keras import regularizers

from tensorflow.keras.models import load_model


import pandas as pd
import os
import urllib.request

import random
import shutil

In [9]:
from classification_models.tfkeras import Classifiers

ResNet18, preprocess_input = Classifiers.get('resnet18')
model_resnet1 = ResNet18(input_shape=(224, 224, 3), weights='imagenet', include_top=False)

# C:/Users/Loser/Documents/vs test projects/emotions_project/FER-2013 BALANCED
test_path=os.path.join('C:/Users/Loser/Documents/vs test projects/emotions_project/FER-2013 BALANCED','test')
# test_path=os.path.join('FER1-2013','test')
test_resnet = image_dataset_from_directory(test_path,batch_size=128,color_mode='rgb',image_size=(224,224))
test_resnet=test_resnet.map(lambda x,y: (preprocess_input(x),y))

model = tf.keras.models.load_model('model_resnet_new_one_again1.keras')

model.evaluate(test_resnet)

Found 3044 files belonging to 7 classes.
24/24 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.6928 - loss: 0.9073


[0.9072874188423157, 0.6928383708000183]

In [3]:
model_name = 'model_resnet_new_one_again1.keras'
url = 'https://www.dropbox.com/scl/fi/quok4ymxq21ip7zdce0h6/model_resnet_new_one_again1.keras?rlkey=glvzbjc1oyktkkaw95nzkdg3u&st=o2i2xr45&dl=1'
if not os.path.exists(model_name):
    urllib.request.urlretrieve(url,model_name)
model = tf.keras.models.load_model('model_resnet_new_one_again1.keras')

In [4]:
def label_translate(label):
    class_names=['angry','disgust','fear','happy','neutral','sad','surprise']
    idx=np.argmax(label)
    return class_names[idx]

# detector=cv.CascadeClassifier('face_detector.xml')  
# print(detector.empty())

In [6]:


# Список эмоций ровно в том порядке, в котором их предсказывает твоя модель model.predict()
EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Глобальный массив, куда мы будем незаметно для основного цикла сохранять историю
analytics_data = []

class CentroidTracker:
    def __init__(self, max_distance=80):
        self.next_id = 1
        self.current_objects = {}  # ID -> (cx, cy)
        self.max_distance = max_distance

    def update(self, faces_rect):
        new_objects = {}
        for box in faces_rect:
            x, y, w, h = box
            cx, cy = int(x + w / 2), int(y + h / 2)
            assigned_id = None
            min_dist = self.max_distance
            
            for obj_id, (prev_cx, prev_cy) in self.current_objects.items():
                dist = np.hypot(cx - prev_cx, cy - prev_cy)
                if dist < min_dist:
                    assigned_id = obj_id
                    min_dist = dist
            
            if assigned_id is None:
                assigned_id = f"Person_{self.next_id}"
                self.next_id += 1
                
            new_objects[assigned_id] = (cx, cy, w, h)
            
        self.current_objects = {obj_id: (b[0], b[1]) for obj_id, b in new_objects.items()}
        return new_objects

def save_frame_analytics(frame_id, obj_id, raw_scores):
    """ Вспомогательная функция, которая упаковывает вероятности в общую базу """
    log_row = {'Frame': frame_id, 'ID': obj_id}
    for i, label in enumerate(EMOTION_LABELS):
        log_row[label] = float(raw_scores[i])
    analytics_data.append(log_row)

def plot_local_results():
    """ Твоя функция визуализации, которая теперь автоматически подтягивает собранные данные """
    if not analytics_data:
        print("Нет данных для графиков.")
        return
        
    df = pd.DataFrame(analytics_data)
    df.to_csv('video_analytics_report.csv', index=False)
    print("Статистика сохранена в 'video_analytics_report.csv'")

    for obj_id in df['ID'].unique():
        person_df = df[df['ID'] == obj_id].sort_values(by='Frame')
        
        plt.figure(figsize=(10, 4))
        for label in EMOTION_LABELS:
            if label in person_df.columns:
                plt.plot(person_df['Frame'], person_df[label], label=label)
        
        plt.title(f"Анализ изменения эмоций для: {obj_id}")
        plt.xlabel("Кадры (Время)")
        plt.ylabel("Уверенность модели (0.0 - 1.0)")
        plt.legend()
        plt.grid(True)
        plt.show()

In [7]:
print('из файла/в реальном времени (нужное Ctrl+C -> Ctrl+V)')
choice=input()

из файла/в реальном времени (нужное Ctrl+C -> Ctrl+V)


In [ ]:
# video_path = r"C:\Users\Loser\Documents\vs test projects\emotions_project\videos_unboxing\Savetik_1776620265.mp4" 

if choice=='из файла':
    video_path = input('Полный путь к файлу: ')
    video_path.replace('\\', '/') 
    # video_path = r"C:\Users\Loser\Documents\vs test projects\emotions_project\videos_unboxing\Savetik_1776620265.mp4" 
    capture = cv.VideoCapture(video_path)
else:
    capture = cv.VideoCapture(0)
detector_path=r"C:\Users\Loser\Documents\vs test projects\emotions_project\face_detector.xml"

detector=cv.CascadeClassifier(detector_path)

tracker=CentroidTracker(max_distance=100)
frame_id=0

if not capture.isOpened():
    print('не смог открыть видеофайл')  
while True:
    is_true, frame = capture.read()

    if not is_true or frame is None:
        print("конец видео или проблема с кадром")
        break
    # print(f'is_true: {is_true}, frame shape: {frame.shape if frame is not None else 'None'}')
    frame_id+=1
    try:
        gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
        faces_rect = detector.detectMultiScale(gray, scaleFactor=1.1,minNeighbors=7)

        tracked = tracker.update(faces_rect)

        for (x,y,w,h) in faces_rect:
            pic = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            pic = cv.resize(frame[y:y+h,x:x+w], (224,224))
            pic = np.expand_dims(pic, axis=0).astype('float32')
            pic=preprocess_input(pic)
            label = model.predict(pic)


            # pic = cv.resize(gray[y:y+h,x:x+w],(75,75))
            # pic=pic/255.0
            # pic = np.expand_dims(pic, axis=0)
            # label = my_model.predict(pic)

            cx, cy = int(x + w / 2), int(y + h / 2)
            current_id = "Unknown"
            for obj_id, (tcx, tcy, tw, th) in tracked.items():
                if abs(cx - tcx) < 20 and abs(cy - tcy) < 20: # если центры совпадают
                    current_id = obj_id
                    break
                    
            save_frame_analytics(frame_id, current_id, label[0])

            # Твой оригинальный выигрышный интерфейс (добавили только ID к тексту)
            display_text = f"Person: {label_translate(label)}"
            cv.putText(frame, text=display_text, org=(x, y-10), fontFace=cv.FONT_HERSHEY_COMPLEX, fontScale=0.5, color=(0, 255, 0), thickness=2)
            cv.rectangle(frame, (x,y),(x+w,y+h),(0,255,0),2)
        cv.imshow('video', frame)
    except Exception as e:
        print(f"косяк с кадрами: {e}")
        break

    # if not is_true:
    #     break

        
    if cv.waitKey(20) & 0xFF==ord('d'):
        break
capture.release()
cv.destroyAllWindows()
plot_local_results()

KeyboardInterrupt: 